# 01 · Data analysis: Prompted Segmentation for Drywall QA

Steps:
0. Setup & load COCO annotations
1. Integrity: unreadable files, size mismatch, empty images, degenerate boxes, exact duplicates
2. Leakage: Roboflow augmented copies (`name.rf.<hash>.jpg`) + near-duplicates (dHash) across splits
3. Rasterize GT masks (polygons → filled; drywall boxes → filled rectangles) + per-object stats
4. Statistics plots (foreground %, objects/img, elongation, orientation, stroke width)
5. Outlier review (full-image masks, tiny objects, rectangle "polygons")
6. Visual sanity grid
7. Scene-variety buckets (brightness / contrast / sharpness) for consistency evaluation later
(the real split is made by `prepare_data.py`)

In [ ]:
%pip install -q pandas

In [ ]:
import json, hashlib, random
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

ROOT = Path("dataset")
WORK = Path("work")
MASKS, FIGS = WORK / "masks", WORK / "figs"
for p in (MASKS, FIGS):
    p.mkdir(parents=True, exist_ok=True)

DATASETS = {
    "taping": {"dir": ROOT / "Drywall-Join-Detect.v2i.coco", "prompt": "segment taping area"},
    "crack":  {"dir": ROOT / "cracks.v1i.coco",              "prompt": "segment crack"},
}
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", None)

## 0. Load annotations
`source` = original filename before Roboflow's `.rf.<hash>` suffix. Images with the same source are
augmented copies of one photo and must never be split across train/test.

In [ ]:
img_rows, ann_rows = [], []
for ds, cfg in DATASETS.items():
    for js in sorted(cfg["dir"].glob("*/_annotations.coco.json")):
        split = js.parent.name
        coco = json.loads(js.read_text(encoding="utf-8"))
        for im in coco["images"]:
            img_rows.append(dict(dataset=ds, rf_split=split, image_id=im["id"],
                                 file=str(js.parent / im["file_name"]),
                                 source=im["file_name"].split(".rf.")[0],
                                 width=im["width"], height=im["height"]))
        for a in coco["annotations"]:
            seg = a.get("segmentation")
            ann_rows.append(dict(dataset=ds, rf_split=split, image_id=a["image_id"], ann_id=a["id"],
                                 bbox=a["bbox"], seg=seg if isinstance(seg, list) else []))

imgs, anns = pd.DataFrame(img_rows), pd.DataFrame(ann_rows)
imgs["key"] = imgs.dataset + "/" + imgs.rf_split + "/" + imgs.image_id.astype(str)
anns["key"] = anns.dataset + "/" + anns.rf_split + "/" + anns.image_id.astype(str)
imgs["n_anns"] = imgs.key.map(anns.groupby("key").size()).fillna(0).astype(int)

print("Images per Roboflow split:")
print(imgs.groupby(["dataset", "rf_split"]).size().unstack(fill_value=0))
print("\nLabel type:")
print(anns.assign(label=np.where(anns.seg.str.len() > 0, "polygon", "box"))
          .groupby(["dataset", "label"]).size().unstack(fill_value=0))

## 1. Integrity
One pass over every image: readability, size check, md5 (exact dupes), dHash (near dupes),
and scene statistics reused in step 7.

In [ ]:
def dhash(gray, n=16):
    """256-bit difference hash: robust to resizing / JPEG re-encoding / colour shifts, not to flips/rotations.
    (64-bit was too coarse: plain walls all hashed alike and chained into one giant group.)"""
    small = cv2.resize(gray, (n + 1, n), interpolation=cv2.INTER_AREA).astype(np.int16)
    return np.packbits((small[:, 1:] > small[:, :-1]).ravel())

recs = []
for r in tqdm(imgs.itertuples(), total=len(imgs), desc="scan"):
    path = Path(r.file)
    raw = path.read_bytes() if path.exists() else b""
    im = cv2.imdecode(np.frombuffer(raw, np.uint8), cv2.IMREAD_COLOR) if raw else None
    if im is None:
        recs.append(dict(readable=False))
        continue
    g = cv2.cvtColor(im, cv2.COLOR_BGR2GRAY)
    recs.append(dict(readable=True,
                     size_ok=(im.shape[1], im.shape[0]) == (r.width, r.height),
                     md5=hashlib.md5(raw).hexdigest(),
                     dhash=dhash(g),
                     brightness=g.mean(), contrast=g.std(),
                     sharpness=cv2.Laplacian(g, cv2.CV_64F).var()))
imgs = imgs.join(pd.DataFrame(recs, index=imgs.index))

anns["bw"] = anns.bbox.map(lambda b: b[2])
anns["bh"] = anns.bbox.map(lambda b: b[3])
anns["degenerate"] = (anns.bw < 2) | (anns.bh < 2)

dup = imgs[imgs.readable & imgs.duplicated("md5", keep=False)]
integrity = pd.DataFrame({
    "images": imgs.groupby("dataset").size(),
    "unreadable": (~imgs.readable).groupby(imgs.dataset).sum(),
    "size_mismatch": (imgs.readable & ~imgs.size_ok.fillna(True).astype(bool)).groupby(imgs.dataset).sum(),
    "no_annotations": (imgs.n_anns == 0).groupby(imgs.dataset).sum(),
    "degenerate_anns": anns.degenerate.groupby(anns.dataset).sum(),
    "exact_dup_files": dup.groupby("dataset").size(),
}).fillna(0).astype(int)
integrity

## 2. Leakage check
(a) Roboflow copies: how many images per original photo, and does one photo appear in several splits?
(b) Near-duplicates: images with different source names but almost identical pixels (dHash Hamming ≤ 4).
Both are merged into **groups** with union-find; the split in step 8 is done per group.

In [ ]:
src = imgs.groupby("dataset").agg(images=("key", "size"), sources=("source", "nunique"))
src["copies_per_source"] = (src.images / src.sources).round(2)
cross = imgs.groupby(["dataset", "source"]).rf_split.nunique()
src["sources_in_>1_rf_split"] = (cross > 1).groupby("dataset").sum()
print(src, "\n")

HAMMING = 12      # of 256 bits (~5%)
MAX_GROUP = 60    # a near-dup merge that would create a bigger group is skipped (stops transitive chaining)
parent, gsize = {}, {}
def find(x):
    if x not in parent:
        parent[x], gsize[x] = x, 0
    while parent[x] != x:
        parent[x] = parent[parent[x]]
        x = parent[x]
    return x
def union(a, b, cap=None):
    ra, rb = find(a), find(b)
    if ra == rb or (cap and gsize[ra] + gsize[rb] > cap):
        return False
    parent[ra] = rb
    gsize[rb] += gsize[ra]
    return True

for k, s in zip(imgs.key, imgs.dataset + "|" + imgs.source):   # same source → same group (always)
    gsize[find(k)] = 1
    union(k, s)

near_pairs = []
for ds, sub in imgs[imgs.readable].groupby("dataset"):
    keys, h = sub.key.to_numpy(), np.stack(sub.dhash.to_numpy())
    srcs, splits = sub.source.to_numpy(), sub.rf_split.to_numpy()
    for i in range(0, len(h), 256):
        d = np.unpackbits(h[i:i + 256, None] ^ h[None], axis=2).sum(2)
        for a, b in zip(*np.nonzero(d <= HAMMING)):
            a += i
            if a < b and srcs[a] != srcs[b]:
                merged = union(keys[a], keys[b], cap=MAX_GROUP)
                near_pairs.append((ds, keys[a], keys[b], splits[a] != splits[b], merged))

near = pd.DataFrame(near_pairs, columns=["dataset", "a", "b", "cross_rf_split", "merged"])
imgs["group"] = imgs.key.map(find)
print(f"near-duplicate pairs with different source names (Hamming ≤ {HAMMING}/256):")
print(near.groupby("dataset").agg(pairs=("a", "size"), cross_rf_split=("cross_rf_split", "sum"),
                                  merged=("merged", "sum")))
gs = imgs.groupby(["dataset", "group"]).size()
print("\ngroups:", gs.groupby("dataset").size().to_dict(), "| largest:", gs.groupby("dataset").max().to_dict())

In [ ]:
def show_pairs(pairs, n=4, title=""):
    if pairs.empty:
        print("no pairs"); return
    rows = pairs.sample(min(n, len(pairs)), random_state=SEED)
    fig, ax = plt.subplots(len(rows), 2, figsize=(6, 3 * len(rows)), squeeze=False)
    path = imgs.set_index("key").file
    for i, r in enumerate(rows.itertuples()):
        for j, k in enumerate((r.a, r.b)):
            ax[i, j].imshow(cv2.cvtColor(cv2.imread(path[k]), cv2.COLOR_BGR2RGB))
            ax[i, j].set_title(k, fontsize=7); ax[i, j].axis("off")
    fig.suptitle(title); plt.tight_layout(); plt.show()

show_pairs(near, title="near-duplicate pairs: confirm these really are the same photo")

## 3. Rasterize GT masks + per-object stats
Masks are saved as single-channel PNG {0,255} at the image's size: `work/masks/<dataset>/<image stem>.png`.

Per object:
- `area_frac`: object area / image area
- `elongation`: long / short side of the min-area rectangle
- `angle`: orientation of the long side (0° = horizontal, 90° = vertical)
- `width_px`: mean stroke width ≈ 2·area / perimeter (accurate for long thin shapes)
- `is_rect`: polygon that is really an axis-aligned rectangle (box disguised as a mask)

In [ ]:
def ann_polys(a):
    polys = [np.asarray(p, np.float32).reshape(-1, 2) for p in a.seg if len(p) >= 6]
    if polys:
        return polys, "polygon"
    x, y, w, h = a.bbox
    return [np.array([[x, y], [x + w, y], [x + w, y + h], [x, y + h]], np.float32)], "box"

def is_axis_rect(polys):
    if len(polys) != 1:
        return False
    p = np.unique(np.round(polys[0], 1), axis=0)
    return len(p) == 4 and len(np.unique(p[:, 0])) == 2 and len(np.unique(p[:, 1])) == 2

size = imgs.set_index("key")[["width", "height"]]
obj_rows = []
for a in anns.itertuples():
    polys, label = ann_polys(a)
    W, H = size.loc[a.key]
    area = sum(abs(cv2.contourArea(p)) for p in polys)
    perim = sum(cv2.arcLength(p, True) for p in polys)
    (_, _), (rw, rh), ang = cv2.minAreaRect(np.concatenate(polys))
    long_, short = max(rw, rh), min(rw, rh)
    angle = (ang if rw >= rh else ang + 90) % 180
    obj_rows.append(dict(label=label, area_frac=area / (W * H),
                         elongation=long_ / max(short, 1), angle=angle,
                         width_px=2 * area / perim if perim else 0,
                         n_points=sum(len(p) for p in polys),
                         is_rect=label == "polygon" and is_axis_rect(polys)))
anns = anns.join(pd.DataFrame(obj_rows, index=anns.index))

ann_by_key = dict(list(anns.groupby("key")))
img_stats = []
for r in tqdm(imgs.itertuples(), total=len(imgs), desc="masks"):
    m = np.zeros((r.height, r.width), np.uint8)
    for a in ann_by_key.get(r.key, pd.DataFrame()).itertuples():
        polys, _ = ann_polys(a)
        cv2.fillPoly(m, [np.round(p).astype(np.int32) for p in polys], 255)
    out = MASKS / r.dataset / (Path(r.file).stem + ".png")
    out.parent.mkdir(parents=True, exist_ok=True)
    cv2.imwrite(str(out), m)
    n_cc = cv2.connectedComponents((m > 0).astype(np.uint8))[0] - 1
    img_stats.append(dict(mask=str(out), fg_frac=(m > 0).mean(), n_components=n_cc))
imgs = imgs.join(pd.DataFrame(img_stats, index=imgs.index))

print(imgs.groupby("dataset").fg_frac.describe(percentiles=[.05, .25, .5, .75, .95]).round(4))
print()
print(anns.groupby("dataset")[["area_frac", "elongation", "width_px"]].median().round(3))

## 4. Statistics plots
The foreground % histogram justifies the Dice/Focal loss (heavy class imbalance);
stroke width shows how thin the cracks are (1 px of error = large IoU drop).

In [ ]:
def orient(a):
    return "vertical" if 60 <= a <= 120 else "horizontal" if a < 30 or a > 150 else "diagonal"
anns["orientation"] = anns.angle.map(orient)

fig, ax = plt.subplots(2, 3, figsize=(16, 8))
specs = [
    (imgs, "fg_frac", "foreground fraction per image", np.linspace(0, 1, 41), False),
    (imgs, "n_anns", "objects per image", np.arange(0, 12) - 0.5, False),
    (anns, "elongation", "object elongation (long/short)", np.logspace(0, 2.5, 40), True),
    (anns, "angle", "object angle (0=horiz, 90=vert)", np.linspace(0, 180, 37), False),
    (anns, "width_px", "stroke width ≈ 2·area/perimeter (px)", np.linspace(0, 120, 41), False),
    (imgs, "brightness", "image brightness (mean gray)", np.linspace(0, 255, 41), False),
]
for axi, (df, col, title, bins, logx) in zip(ax.ravel(), specs):
    for ds, sub in df.groupby("dataset"):
        axi.hist(sub[col], bins=bins, alpha=0.55, density=True, label=ds)
    axi.set_title(title)
    if logx:
        axi.set_xscale("log")
    axi.legend()
plt.tight_layout()
plt.savefig(FIGS / "stats.png", dpi=120)
plt.show()

print(pd.crosstab(anns.dataset, anns.orientation, normalize="index").round(3))
print("\nthin objects (width < 5 px):", (anns.width_px < 5).groupby(anns.dataset).mean().round(3).to_dict())

## 5. Outlier review
Flags per image. Look at the grids and decide keep / fix / drop for each flag. Record the decision in the README.

In [ ]:
rect_keys = set(anns.loc[anns.is_rect, "key"])
tiny_keys = set(anns.loc[anns.area_frac < 0.001, "key"])
degen_keys = set(anns.loc[anns.degenerate, "key"])
imgs["flag_full_mask"] = imgs.fg_frac > 0.95
imgs["flag_huge_mask"] = imgs.fg_frac.between(0.7, 0.95)
imgs["flag_empty"] = imgs.fg_frac == 0
imgs["flag_tiny_obj"] = imgs.key.isin(tiny_keys)
imgs["flag_rect_poly"] = imgs.key.isin(rect_keys)
imgs["flag_degenerate"] = imgs.key.isin(degen_keys)
FLAGS = [c for c in imgs.columns if c.startswith("flag_")]
imgs.groupby("dataset")[FLAGS].sum()

In [ ]:
def overlay(img_bgr, mask, color=(255, 0, 0)):
    rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    out = rgb.copy()
    out[mask > 0] = (0.55 * out[mask > 0] + 0.45 * np.array(color)).astype(np.uint8)
    cnts, _ = cv2.findContours((mask > 0).astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    cv2.drawContours(out, cnts, -1, (255, 255, 0), 1)
    return out

def grid(df, n=8, cols=4, title="", save=None):
    if df.empty:
        print(f"{title}: none"); return
    rows = df.sample(min(n, len(df)), random_state=SEED)
    r_ = int(np.ceil(len(rows) / cols))
    fig, ax = plt.subplots(r_, cols, figsize=(3.2 * cols, 3.2 * r_), squeeze=False)
    for axi in ax.ravel():
        axi.axis("off")
    for axi, r in zip(ax.ravel(), rows.itertuples()):
        axi.imshow(overlay(cv2.imread(r.file), cv2.imread(r.mask, cv2.IMREAD_GRAYSCALE)))
        axi.set_title(f"{r.dataset} fg={r.fg_frac:.1%} n={r.n_anns}", fontsize=8)
    fig.suptitle(title)
    plt.tight_layout()
    if save:
        plt.savefig(FIGS / save, dpi=110)
    plt.show()

for flag in FLAGS:
    for ds in DATASETS:
        grid(imgs[(imgs.dataset == ds) & imgs[flag]], title=f"{ds}: {flag}")

## 6. Visual sanity grid (random samples)

In [ ]:
for ds in DATASETS:
    grid(imgs[imgs.dataset == ds], n=16, title=f"{ds}: random samples", save=f"samples_{ds}.png")

## 7. Scene-variety buckets
Tercile buckets per dataset. In evaluation, report IoU per bucket. A small spread across
buckets is the evidence for the **Consistency (30 pts)** criterion.

In [ ]:
for col, names in [("brightness", ["dark", "mid", "bright"]),
                   ("contrast", ["low", "mid", "high"]),
                   ("sharpness", ["blurry", "mid", "sharp"])]:
    imgs[f"{col}_bucket"] = imgs.groupby("dataset")[col].transform(
        lambda s: pd.qcut(s.rank(method="first"), 3, labels=names)).astype(str)

for ds in DATASETS:
    sub = imgs[imgs.dataset == ds]
    fig, ax = plt.subplots(3, 4, figsize=(12, 9))
    for i, b in enumerate(["dark", "mid", "bright"]):
        pick = sub[sub.brightness_bucket == b].sample(4, random_state=SEED)
        for j, r in enumerate(pick.itertuples()):
            ax[i, j].imshow(overlay(cv2.imread(r.file), cv2.imread(r.mask, cv2.IMREAD_GRAYSCALE)))
            ax[i, j].set_title(f"{b} ({r.brightness:.0f})", fontsize=8)
            ax[i, j].axis("off")
    fig.suptitle(f"{ds}: brightness buckets")
    plt.tight_layout()
    plt.show()

## Split
The train/val/test split used for training is made by `prepare_data.py` (grouped by source photo,
70/15/15, seed 42), not in this notebook. This notebook is only for understanding the data.
